In [1]:
# !pip install scikeras

# # Uninstall the current scikit-learn version
# !pip uninstall scikit-learn -y

# # Install a compatible version of scikit-learn (e.g., 1.4.2)
# !pip install scikit-learn==1.4.2

Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 124.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
cuml-cu12 26.2.0 requires scikit-learn>=1.5, but you have scikit-learn 1.4.2 which is incompatible.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from keras.callbacks import EarlyStopping

from scikeras.wrappers import KerasClassifier

from sklearn.calibration import CalibrationDisplay
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer,IterativeImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    multilabel_confusion_matrix,
    f1_score
)
from sklearn.utils.class_weight import compute_class_weight

In [3]:
# Read training data
training_df = pd.read_csv(
    filepath_or_buffer='training_faults_diagnostics.csv',
    low_memory=False
)

In [4]:
# Convert SPN and FMI values to strings
training_df['spn'] = training_df['spn'].astype(str)
training_df['fmi'] = training_df['fmi'].astype(str)
print(f'spn datatype: {training_df['spn'].dtype}')

spn datatype: object


In [5]:
target = 'Derate_Target'

# Create dataset with features
X = training_df

# Create array of targets
y = training_df[target]

labels = np.unique(y)

## Identify features for imputing missing values

In [6]:
# Group categorical columns
categorical_columns = ['EquipmentID', 'spn', 'fmi', 'active', 'Severity_Level']

# Group numeric columns
numeric_columns = [
    'BarometricPressure',
    'EngineCoolantTemperature',
    'EngineLoad',
    'EngineOilPressure',
    'EngineOilTemperature',
    'EngineRpm',
    'FuelRate',
    'FuelTemperature',
    'IntakeManifoldTemperature',
    'Speed',
    'Throttle',
    'TurboBoostPressure'
  ]

## Split training dataset

In [7]:
random_state = 42

# Split training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=random_state,
    stratify=y
)

# Split training and validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.1,
    random_state=random_state,
    stratify=y
)

## Create pipeline and fit model

In [8]:
# Initialize pipeline for transforming categorical columns
categorical_pipe = Pipeline(
    steps=[
        ('categorical_imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]
)

# Initialize pipeline for transforming numeric columns
numeric_pipe = Pipeline(
    steps=[
        ('numeric_imputer', IterativeImputer(max_iter=5, random_state=random_state)),
        ('scaler', StandardScaler())
    ]
)

In [9]:
# Initialize column transformer
ct = ColumnTransformer(
    transformers=[
        ('categorical_pipe', categorical_pipe, categorical_columns),
        ('numeric_pipe', numeric_pipe, numeric_columns)
    ]
)

In [10]:
# Get the number of features after one hot encoding
ct.fit(X_train)
X_val_transform = ct.transform(X_val)
n_features = ct.transform(X_train[:1]).shape[1]
print(f'n_features: {n_features}')

# Transform train labels
y_train_transform = tf.keras.utils.to_categorical(
    x=y_train.values,
    num_classes=len(labels)
)

# Transform validation labels
y_val_transform = tf.keras.utils.to_categorical(
    y_val.values,
    num_classes=len(labels)
)

# Transform test labels
y_test_transform = tf.keras.utils.to_categorical(
    x=y_test.values,
    num_classes=len(labels)
)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:801: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


n_features: 1412


In [101]:
# Initialize early stopping
es = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# Initialize class weights
# weights = compute_class_weight(
#     class_weight='balanced',
#     classes=labels,
#     y=y_train
# )
# class_weights = dict(zip(labels, weights))
# {0: 0.33392947189837874, 1: 397.27242386316226, 2: 352.29818719940806}

class_weights = {
    0: 1,
    1: 2,
    2: 1
}
class_weights

{0: 1, 1: 2, 2: 1}

In [102]:
# Function to create the Keras model for SciKeras
def create_model():
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(shape=(n_features,)))
    model.add(tf.keras.layers.Dense(64, activation='relu'))
    model.add(tf.keras.layers.Dense(32, activation='relu'))
    model.add(tf.keras.layers.Dense(3, activation='softmax'))
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=[
            tf.keras.metrics.Precision(),
            tf.keras.metrics.Recall()
        ]
    )
    return model

# Keras model with SciKeras wrapper
model = KerasClassifier(
    model=create_model,
    callbacks=[es],
    epochs=20,
    batch_size=32
)

In [103]:
# Initialize pipeline for training model
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('model', model)
    ]
)

In [104]:
# Fit the model with training data, encoded labels, and validation data
pipe.fit(
    X=X_train,
    y=y_train_transform,
    model__validation_data=(X_val_transform, y_val_transform),
    model__class_weight=class_weights
)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:801: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Epoch 1/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 74s 2ms/step - loss: 0.0131 - precision_9: 0.9985 - recall_6: 0.9975 - val_loss: 0.0084 - val_precision_9: 0.9986 - val_recall_6: 0.9984
Epoch 2/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 69s 2ms/step - loss: 0.0105 - precision_9: 0.9986 - recall_6: 0.9984 - val_loss: 0.0075 - val_precision_9: 0.9986 - val_recall_6: 0.9984
Epoch 3/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 68s 2ms/step - loss: 0.0102 - precision_9: 0.9986 - recall_6: 0.9984 - val_loss: 0.0074 - val_precision_9: 0.9986 - val_recall_6: 0.9985
Epoch 4/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 68s 2ms/step - loss: 0.0099 - precision_9: 0.9986 - recall_6: 0.9984 - val_loss: 0.0071 - val_precision_9: 0.9987 - val_recall_6: 0.9985
Epoch 5/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 68s 2ms/step - loss: 0.0096 - precision_9: 0.9987 - recall_6: 0.9984 - val_loss: 0.0069 - val_precision_9: 0.9987 - val_recall_6: 0.9985
Epoch 6/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 68s 2ms/step - loss: 0.0096 - precision_9: 0.9987 -

Pipeline(steps=[('transformer',
                 ColumnTransformer(transformers=[('categorical_pipe',
                                                  Pipeline(steps=[('categorical_imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['EquipmentID', 'spn', 'fmi',
                                                   'active',
                                                   'Severity_Level']),
                                                 ('numeric_pipe',
                                                  Pipeline(steps=[('numeric_imputer',
                                                                   IterativeImputer(max_iter=5,
                                                                                    rand...
                                                   'EngineLoad',
                                                   'EngineOilPressure',
                                                   'EngineOilTemperature',
                                                   'EngineRpm', 'FuelRate',
                                                   'FuelTemperature',
                                                   'IntakeManifoldTemperature',
                                                   'Speed', 'Throttle',
                                                   'TurboBoostPressure'])])),
                ('model',
                 KerasClassifier(batch_size=32, callbacks=[<keras.src.callbacks.early_stopping.EarlyStopping object at 0x79ca6a1bfce0>], epochs=20, model=<function create_model at 0x79c988593880>))])

## Compare Training and Testing

In [105]:
# Predict training and testing data
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

29759/29759 ━━━━━━━━━━━━━━━━━━━━ 66s 2ms/step
6613/6613 ━━━━━━━━━━━━━━━━━━━━ 16s 2ms/step


In [106]:
# Training classification report
training_cr = classification_report(
    y_true=y_train_transform,
    y_pred=y_pred_train,
    digits=6
)
print(str(training_cr))

# Training confusion matrix
training_cm = confusion_matrix(
    y_true=y_train.values, # Use original integer labels for y_true
    y_pred=np.argmax(y_pred_train, axis=1) # Convert one-hot encoded predictions to integer labels
)
print(training_cm)

              precision    recall  f1-score   support

           0   0.998873  0.999803  0.999338    950562
           1   0.484536  0.117647  0.189325       799
           2   0.710145  0.489456  0.579501       901

   micro avg   0.998580  0.998580  0.998580    952262
   macro avg   0.731185  0.535635  0.589388    952262
weighted avg   0.998169  0.998580  0.998261    952262
 samples avg   0.998580  0.998580  0.998580    952262

[[950375     64    123]
 [   648     94     57]
 [   424     36    441]]


In [107]:
# Testing classification report
testing_cr = classification_report(
    y_true=y_test_transform,
    y_pred=y_pred_test,
    digits=6
)
print(str(testing_cr))

# Testing confusion matrix
testing_cm = confusion_matrix(
    y_true=y_test.values, # Use original integer labels for y_true
    y_pred=np.argmax(y_pred_test, axis=1) # Convert one-hot encoded predictions to integer labels
)
print(testing_cm)

              precision    recall  f1-score   support

           0   0.998851  0.999839  0.999345    211236
           1   0.476190  0.112360  0.181818       178
           2   0.708661  0.450000  0.550459       200

   micro avg   0.998573  0.998573  0.998573    211614
   macro avg   0.727901  0.520733  0.577207    211614
weighted avg   0.998137  0.998573  0.998233    211614
 samples avg   0.998573  0.998573  0.998573    211614

[[211202     14     20]
 [   141     20     17]
 [   102      8     90]]


## Predict Unseen Data

In [108]:
testing_df = pd.read_csv(
    filepath_or_buffer='testing_faults_diagnostics.csv',
    low_memory=False
)

# Convert SPN and FMI values to strings
testing_df['spn'] = testing_df['spn'].astype(str)
testing_df['fmi'] = testing_df['fmi'].astype(str)
print(f'spn datatype: {testing_df['spn'].dtype}')

unseen_data = testing_df.drop(columns=['Derate_Target'])
y_unseen = testing_df['Derate_Target']

spn datatype: object


In [109]:
# Transform unseen labels
y_unseen_transform = tf.keras.utils.to_categorical(
    x=y_unseen.values,
    num_classes=3
)

In [110]:
# Predict unseen data
y_pred_unseen = pipe.predict(unseen_data)

4040/4040 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step


In [111]:
# Unseen classification report
unseen_cr = classification_report(
    y_true=y_unseen_transform,
    y_pred=y_pred_unseen,
    digits=6
)
print(str(unseen_cr))

# Unseen confusion matrix
unseen_cm = confusion_matrix(
    y_true=y_unseen.values, # Use original integer labels for y_true
    y_pred=np.argmax(y_pred_unseen, axis=1) # Convert one-hot encoded predictions to integer labels
)
print(unseen_cm)

              precision    recall  f1-score   support

           0   0.998820  0.984454  0.991585    128970
           1   0.039921  0.411168  0.072776       197
           2   0.081967  0.101010  0.090498        99

   micro avg   0.982903  0.982903  0.982903    129266
   macro avg   0.373569  0.498877  0.384953    129266
weighted avg   0.996656  0.982903  0.989494    129266
 samples avg   0.982903  0.982903  0.982903    129266

[[126965   1895    110]
 [   114     81      2]
 [    36     53     10]]
